# Session 13. Deep agents: planning, files, subagents

**Claude Code's playbook, shipped as a package.**

- four moves: a long prompt, a todo list, a filesystem, subagents
- the harness-engineering notebook (self-study reading) reads them out of Claude Code
- `create_deep_agent` returns the plain LangGraph graph you already know
- materials pin deepagents 0.6.x; the 0.7 alphas churn weekly


In [ ]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()  # reads .env once; nothing below opens a file


def chat_model(size: str = "cheap", **kwargs):
    """A model object for the configured provider. A dozen lines, copy them once."""
    name = os.environ[f"MODEL_{size.upper()}"]  # ids live in .env, never in code
    secret = os.environ["LLM_API_KEY"]
    if os.getenv("LLM_REASONING_EFFORT"):  # gpt-5.x: tools need reasoning "none"
        kwargs.setdefault("reasoning_effort", os.environ["LLM_REASONING_EFFORT"])
    # Gemini over OpenAI-compat drops the reasoning signature: turn two 400s
    if os.getenv("LLM_PROVIDER", "openai_compat") == "google_genai":
        return init_chat_model(f"google_genai:{name}", api_key=secret, **kwargs)
    return init_chat_model(
        f"openai:{name}", api_key=secret, base_url=os.environ["LLM_BASE_URL"], **kwargs
    )


print("provider:", os.getenv("LLM_PROVIDER", "openai_compat"),
      "| strong:", os.environ["MODEL_STRONG"])


## From dissection to package

**The harness-engineering notebook (self-study) read four pillars out of Claude Code.**

- a long base prompt, a planning tool, a filesystem, a subagent spawner
- deepagents ships each pillar as a middleware over the `create_agent` core from session 3
- today's vehicle: a model council for debatable study questions


**Four pillars became five: 0.6.12 adds skills.**

- a skill is a `SKILL.md` on the agent's filesystem: name, description, then instructions
- `create_deep_agent(skills=["/skills/"])` names source dirs; `SkillsMiddleware` builds the roster
- name and description ride the system prompt, Claude Code's own mechanic; bodies load on demand
- proof without a single model call, right after the solo tour


In [ ]:
from deepagents import create_deep_agent

QUESTION = "Should a beginner learn recursion before loops?"

SOLO_PROMPT = """You research debatable study questions.
Plan with the todo list, keep notes in files, then answer in two sentences."""

solo = create_deep_agent(model=chat_model("strong"), system_prompt=SOLO_PROMPT)

print(type(solo).__name__)  # the same class every graph so far compiled to
print(list(solo.nodes))
print(sorted(solo.nodes["tools"].bound.tools_by_name))  # the free toolset


**Nine tools you never passed. Three of the pillars hide in them.**

- `write_todos` is the planning tool; the plan lives in `state["todos"]`
- `ls`, `read_file`, `write_file`, `edit_file`, `glob`, `grep`: a filesystem in state, not on disk
- `task` spawns subagents; `execute` stays inert without a sandbox backend
- no delete tool on 0.6.x: that arrives with the 0.7 renames


In [ ]:
from deepagents.graph import BASE_AGENT_PROMPT

print(len(BASE_AGENT_PROMPT), "chars of pillar four, in the box")
print(BASE_AGENT_PROMPT[:180])  # your system_prompt is prepended, never substituted


**The stack is middleware you have met, in a fixed order.**

Base first, then yours, then the tail. `middleware=` adds between them; it replaces nothing.

```text
TodoList -> Filesystem -> SubAgent -> Summarization -> PatchToolCalls
        -> [your middleware] -> tool exclusion, caching, memory, HITL
```

- `SummarizationMiddleware` from session 4 is already inside
- swapping a default out is a harness-profile feature, not `middleware=`


In [ ]:
state = solo.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 20},  # explicit budget on every invoke
)

print(state["todos"])
print(sorted(state["files"]))  # paths in graph state; the disk is untouched
print(next(iter(state["files"].values())))  # one entry: content plus timestamps
print(state["messages"][-1].content)


**One prompt line each, and every pillar answered.**

- the plan landed in `state["todos"]`, the notes in `state["files"]`
- both are state channels beside `messages`: checkpoint them, inspect them, diff them
- a file entry carries content plus timestamps; nothing touched your machine


In [ ]:
# only hook middleware become nodes; the rest wrap the model call
print(solo.get_graph().draw_mermaid())


In [ ]:
# the fifth pillar, live: mount a skill, see it, call no model
from langchain_core.language_models import GenericFakeChatModel
from langgraph.checkpoint.memory import InMemorySaver

from deepagents.backends.utils import create_file_data

FLASHCARDS = """---
name: flashcards
description: Turns any study topic into three question-answer cards.
---
Write three cards for the topic, hardest concept first.
Phrase every question so it has one checkable answer.
"""

skilled = create_deep_agent(
    model=GenericFakeChatModel(messages=iter([])),  # empty fake: a call would raise
    skills=["/skills/"],  # source dirs live on the agent's filesystem
    checkpointer=InMemorySaver(),
)
print([node for node in skilled.nodes if "Skills" in node])

peek = {"configurable": {"thread_id": "skills-peek"}}
skilled.invoke(
    {"messages": [{"role": "user", "content": "cards on recursion"}],
     "files": {"/skills/flashcards/SKILL.md": create_file_data(FLASHCARDS)}},
    config=peek,
    interrupt_before=["model"],  # session 6 machinery: halt before the model node
)
for skill in skilled.get_state(peek).values["skills_metadata"]:
    print(skill["name"], "|", skill["description"], "|", skill["path"])
print(skilled.get_state(peek).next)  # parked: the model node never ran


## The council

**One model asserting is an anecdote. A ranked anonymous panel is a measurement.**

- Karpathy's llm-council: members answer, peers rank blind, a chair synthesizes
- we rebuild it as one deep agent with five declared subagents, plus one the package adds itself
- stage 1 writes files, stage 2 ranks them, stage 3 reads both


In [ ]:
cheap_a = chat_model("cheap")  # one object per seat, created once and reused
cheap_b = chat_model("cheap")
strong_seat = chat_model("strong")

PERSONAS = [
    (cheap_a, "You weigh cognitive load: what fits in a novice's head today."),
    (cheap_b, "You weigh industry practice: what working code actually does."),
    (strong_seat, "You weigh theory: what generalizes furthest from first principles."),
]

MEMBER_RULES = """Answer the study question in at most three sentences.
Write the answer to the exact file path given in the task, then stop.
Never mention your name, your model, or the other members."""

members = [
    {
        "name": f"member-{n}",
        "description": f"Council seat {n}, fixed persona, answers study questions.",
        "system_prompt": f"{view}\n{MEMBER_RULES}",
        "model": model,
    }
    for n, (model, view) in enumerate(PERSONAS, start=1)
]
print([m["name"] for m in members], "| two cheap seats, one strong")


In [ ]:
from pydantic import BaseModel, Field


class Ranking(BaseModel):
    """One reviewer's blind verdict over the council files."""

    best: str = Field(description="path of the strongest answer")
    worst: str = Field(description="path of the weakest answer")
    reason: str = Field(description="one sentence, no guesses about authorship")


REVIEWER_PROMPT = """List /council with ls and read every answer file there.
Rank the answers on clarity for a beginner.
Return a Ranking. No prose, no guesses about who wrote what."""

reviewer_a = chat_model("cheap")
reviewer_b = chat_model("cheap")

reviewers = [
    {"name": "reviewer-1", "description": "Blind-ranks the council answers.",
     "system_prompt": REVIEWER_PROMPT, "model": reviewer_a,
     "response_format": Ranking},
    {"name": "reviewer-2", "description": "Second, independent blind ranking.",
     "system_prompt": REVIEWER_PROMPT, "model": reviewer_b,
     "response_format": Ranking},
]
print(Ranking.model_json_schema()["required"])


**A subagent is a dict. The interesting key is `model`.**

- required: `name`, `description`, `system_prompt`; the chair reads only the description
- `model=` takes a provider string or any chat-model object: mixed seats, one council
- `response_format=` with a Pydantic class turns a subagent into a typed function
- deepagents auto-adds a `general-purpose` seat: it rides in the roster, the chair prompt never calls it


In [ ]:
CHAIR_PROMPT = """You chair a council answering debatable study questions.
Stage 1: task all three members in one message; tell member N to write
an unsigned answer to /council/answer_N.md, numbered in task order.
Stage 2: once every answer exists, task both reviewers to rank the files.
Stage 3: read the winning file, then write a verdict in two sentences.
Never reveal, in any file or in the verdict, who wrote which answer."""

chair_model = chat_model("strong")
chair = create_deep_agent(
    model=chair_model,
    subagents=[*members, *reviewers],  # dicts in, the task tool learns the roster
    system_prompt=CHAIR_PROMPT,
)

roster = chair.nodes["tools"].bound.tools_by_name["task"].description
start = roster.index("Available agent types")
print(roster[start : roster.index("\n\nWhen using", start)])


In [ ]:
council_run = chair.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 40},  # wide budget: five subagents inside
)

for path, entry in sorted(council_run["files"].items()):
    print(path, "->", entry["content"].split(".")[0])

rankings = [msg.content for msg in council_run["messages"]
            if getattr(msg, "name", None) == "task" and msg.content.startswith("{")]
print(*rankings, sep="\n")  # raw JSON: response_format kept its promise
print("\nverdict:", council_run["messages"][-1].content)


**Read the chair's transcript. Count what is not in it.**

- one `ToolMessage` per subagent: the summary came back, the internals did not
- members spent their own context on files; the chair paid one message per seat
- one shared `files` channel: members wrote it, reviewers and the chair read the very same dict
- the chair knows the author mapping; the reviewers never saw it


In [ ]:
for msg in council_run["messages"]:
    calls = ",".join(call["name"] for call in getattr(msg, "tool_calls", []) or [])
    origin = getattr(msg, "name", None) or calls or "-"
    print(f"{type(msg).__name__:13} {origin:15} {str(msg.content)[:44]!r}")


## What the prompt cannot promise

**Two known failure modes, and a deterministic checker for each.**

- stage skipping: the chair answers without convening anyone
- authorship leaks: a name in a file or verdict un-blinds the reviewers
- live runs fail these sometimes; the checkers tell the truth every time


In [ ]:
MEMBER_NAMES = [m["name"] for m in members]
REVIEWER_NAMES = [r["name"] for r in reviewers]


def stages_run(messages) -> bool:
    """PASS when three member task calls precede two reviewer task calls."""
    seats = [call["args"]["subagent_type"]
             for msg in messages
             for call in getattr(msg, "tool_calls", []) or []
             if call["name"] == "task"]
    hired = [s for s in seats if s in MEMBER_NAMES]  # stage-1 seats, in call order
    ranked = [s for s in seats if s in REVIEWER_NAMES]  # stage-2 seats
    first_rank = min((i for i, s in enumerate(seats) if s in REVIEWER_NAMES),
                     default=len(seats))
    last_hire = max((i for i, s in enumerate(seats) if s in MEMBER_NAMES),
                    default=-1)
    ok = len(set(hired)) == 3 and len(set(ranked)) == 2 and last_hire < first_rank
    print("stages_run:", "PASS" if ok else
          f"FAIL ({len(hired)} member calls, {len(ranked)} reviewer calls)")
    return ok


def leak_scan(files, verdict) -> bool:
    """PASS when no member name appears in any answer or in the verdict."""
    text = verdict + "".join(entry["content"] for entry in files.values())
    leaks = sorted(name for name in MEMBER_NAMES if name in text)
    print("leak_scan:", f"FAIL {leaks}" if leaks else "PASS")
    return not leaks


stages_run(council_run["messages"])
leak_scan(council_run["files"], council_run["messages"][-1].content)


In [ ]:
lax_model = chat_model("strong")
lax_chair = create_deep_agent(
    model=lax_model,
    subagents=[*members, *reviewers],  # same council, ceremony left to goodwill
    system_prompt="You answer study questions well.",
)

lax_run = lax_chair.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 40},
)
print(lax_run["messages"][-1].content)
stages_run(lax_run["messages"])  # the skipped stage is a printed fact now
leak_scan(lax_run["files"], lax_run["messages"][-1].content)


## Prompt orchestration vs code orchestration

**A prompt requests an order. A graph enforces one.**

- the staged chair usually complies; nothing but the checkers holds it there
- while stage order is a preference, prompt away: flexible, cheap to change
- once stage order is a contract, it belongs in edges: session 9 machinery


In [ ]:
from langchain_core.messages import AIMessage
from langgraph.graph import END, START, MessagesState, StateGraph


def hire(state: MessagesState) -> dict:  # real nodes would invoke members here
    return {"messages": [AIMessage(content="stage 1: three answers collected")]}


def rank(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="stage 2: two rankings collected")]}


def decide(state: MessagesState) -> dict:
    return {"messages": [AIMessage(content="stage 3: verdict written")]}


builder = StateGraph(MessagesState)
builder.add_node("hire", hire)
builder.add_node("rank", rank)
builder.add_node("decide", decide)
builder.add_edge(START, "hire")
builder.add_edge("hire", "rank")  # the stage order is now a compile-time fact
builder.add_edge("rank", "decide")
builder.add_edge("decide", END)
pipeline = builder.compile()  # reordering stages now means recompiling

strict_chair = create_deep_agent(
    model=chat_model("strong"),
    subagents=[{"name": "council-pipeline",
                "description": "Runs the whole council, stages hardwired.",
                "runnable": pipeline}],
    system_prompt="Delegate the whole question to council-pipeline.",
)
strict_run = strict_chair.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 20},
)
print(strict_run["messages"][-2].content)  # the pipeline's report, one ToolMessage


**Same `task` call from outside. No obedience needed inside.**

- `CompiledSubAgent` is `{name, description, runnable}` around any compiled graph
- the chair still delegates by prompt; the stages inside cannot reorder
- stub nodes today; your real nodes run members and reviewers in code


## Where the money went, per seat

**One council run becomes one trace tree in Langfuse.**

- every `task` span carries `lc_agent_name` metadata: group cost per member
- compare cost and latency of the strong seat against the cheap ones
- needs the session-2 Langfuse stack up; in the trace, `lc_agent_name` sits on every member's spans


In [ ]:
import os

from langfuse import get_client
from langfuse.langchain import CallbackHandler

client = get_client()  # reads LANGFUSE_HOST and both keys from the environment
print("server:", os.getenv("LANGFUSE_HOST"), "| up:", client.auth_check())

handler = CallbackHandler()  # a fresh handler per run gives one trace per run
traced = chair.invoke(
    {"messages": [{"role": "user", "content": QUESTION}]},
    config={"recursion_limit": 40, "callbacks": [handler]},
)

client.flush()  # a notebook kernel never exits, so nothing sends without this

print(traced["messages"][-1].content)


## Practice

**Your own council, in your own repository.**

1. pick a debatable question from your project's domain, not this notebook's
2. three members with distinct personas; any chat-model object plugs into `model=`
3. two reviewers returning your own `Ranking`; the chair prompt fixes `answer_N.md` naming
4. run it: both checkers must print PASS over the transcript and files


**Then un-blind it on purpose, once.**

5. rerun with member names written inside the files; watch the rankings move
6. that is session 11's evaluator bias, reproduced on your own agent
7. keep the blind version; the leak checker guards it from now on


**Required artifact: `runs/session-13.md`, committed.**

- the `files` listing, both raw ranking JSONs, and the verdict
- one sentence naming the session-11 bias that blinding removed
- the exported council trace, committed next to it: one tree, five subagent spans
- stretch: move one stage into a `CompiledSubAgent` and show the order held


**Today at the desk: eval-test acceptance.**

- bring the session-11 set: dataset, trajectory test, judge, threshold, three runs
- sign-off, or a fix list due next session; the checkpoint terms live in the grading document, docs/04-grading.md
- deepagents is an accepted project base once `create_agent` runs out of room
- optional self-study: the deep-agents-from-scratch repository


## Today, in one card

**A deep agent is `create_agent` plus four middlewares: a long prompt, a todo list, a filesystem in state, subagents.**

**You can now defend:**
- the plan lives in `state["todos"]` and the notes in `state["files"]`: state channels beside `messages`, and nothing touches your disk
- a subagent returns one `ToolMessage`; its transcript stays in its own context, and the chair pays one message per seat
- a prompt requests a stage order and only edges enforce one, so a contract belongs in a `CompiledSubAgent` graph

**In your repository:** `runs/session-13.md`, the files listing, both ranking JSONs, the verdict, the bias that blinding removed, the council trace.
**The trap of the day:** the chair can skip a stage or leak authorship, and a live run sometimes does; the checkers, not the prompt, tell the truth.
**Ask yourself:** one model asserting is an anecdote and a blind ranked panel is a measurement: which session-11 bias does the blinding remove?
**Next time:** the same tools arrive over a protocol: MCP, and other people's servers.